# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show metadata summary
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print("Keywords:", ', '.join(dataset.metadata.keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs. List the available record sets and their fields using the `@id` identifiers.

If there are multiple record sets, we list each. The record sets and fields are referenced by their `@id` throughout this notebook.

In [ ]:
# List all record sets by @id and their fields' @id
record_sets = dataset.metadata.recordSet
record_set_ids = []
if record_sets:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        record_set_ids.append(rs['@id'])
        if 'field' in rs:
            for field in rs['field']:
                print(f"  Field @id: {field['@id']} | Name: {field.get('name', '')} | DataType: {field.get('dataType', '')}")
else:
    print("No recordSet defined in metadata.")

# For demonstration, attempt to list a few records for each record set, if any
for rs_id in record_set_ids:
    print(f"\nSample records from RecordSet {rs_id}:")
    try:
        for idx, record in enumerate(dataset.records(record_set=rs_id)):
            print(record)
            if idx > 2:
                break
    except Exception as e:
        print(f"Cannot load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We extract the data from each record set. All objects are referenced by their `@id`, as required.

In [ ]:
# Extract data from each record set using @id
dataframes = {}

# If no recordSet is in metadata, we attempt to load from the default record set
if record_set_ids:
    for rs_id in record_set_ids:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for RecordSet {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns")
            print("Columns:", df.columns.tolist())
else:
    # Try loading the default recordSet if none listed
    default_rs_id = dataset.metadata['@id']
    print(f"No explicit recordSet found; using dataset @id: {default_rs_id}")
    records = list(dataset.records(record_set=default_rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[default_rs_id] = df
        print(f"Loaded DataFrame for RecordSet {default_rs_id}: {df.shape[0]} rows, {df.shape[1]} columns")
        print("Columns:", df.columns.tolist())
    else:
        print("No records found.")

# Show head of one sample DataFrame
sample_rs_id = next(iter(dataframes.keys()), None)
if sample_rs_id:
    dataframes[sample_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations include removing outliers, transforming distributions, grouping data by key attributes.

**All fields/columns are referenced by `@id`.**

In [ ]:
from numpy import nan

# Pick the sample record set
record_set_id = sample_rs_id
df = dataframes[record_set_id]

# Find a numeric column by @id or name
numeric_col_id = None
for col in df.columns:
    # Try to locate typical numeric log-likelihood, coefficient, or p-value columns
    if "log_likelihood" in col.lower() or "coef" in col.lower() or "p_value" in col.lower():
        numeric_col_id = col
        break
if not numeric_col_id:
    # Fallback: try first float/integer column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col_id = col
            break
print(f"Numeric field chosen: {numeric_col_id}")

# Filtering: remove missing values and filter values above a threshold
threshold = 10
filtered_df = df[df[numeric_col_id].fillna(-9999) > threshold]
print(f"Filtered records with {numeric_col_id} > {threshold}:")
print(filtered_df.head())

# Normalization
col_norm = f"{numeric_col_id}_normalized"
filtered_df[col_norm] = (filtered_df[numeric_col_id] - filtered_df[numeric_col_id].mean()) / filtered_df[numeric_col_id].std()
print(f"Normalized {numeric_col_id} for filtered records:")
print(filtered_df[[numeric_col_id, col_norm]].head())

# Group by another field (@id)
group_field_id = None
for col in df.columns:
    if "ward" in col.lower() or "county" in col.lower() or "gender" in col.lower():
        group_field_id = col
        break
print(f"Group field chosen: {group_field_id}")
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_col_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_col_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Here we visualize the distribution of the selected numeric variable (referenced by its `@id`), plus grouped means if available.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_col_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_col_id}")
plt.xlabel(numeric_col_id)
plt.ylabel("Frequency")
plt.show()

if group_field_id:
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_col_id)
    plt.title(f"Mean of {numeric_col_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The [FAIR² dataset](https://doi.org/10.71728/senscience.y7m0-f273) contains ordered logistic regression results examining predictors of adoption for indigenous and modern knowledge in rangeland management in Northern Kenya.
- We loaded the dataset using the Croissant schema and explored the available record sets and fields using `@id` references.
- A numeric field (e.g., log-likelihood or coefficient) was selected and analyzed: records filtered above a threshold, normalized, and grouped by a demographic or geographic attribute.
- Visualizations illustrated the distribution and group means, showing variation by ward, county, or gender.
- The dataset is suitable for policy analysis and community intervention planning but contains potential biases and missingness as described in the metadata.

Further work: deeper domain analysis, modeling, or fairness evaluation may be performed next, referencing fields/columns by their Croissant `@id` for consistency.